In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
import os
from dotenv import load_dotenv

In [2]:
#Load API key
load_dotenv(dotenv_path=".env")
openai_api_key=os.getenv("OPENAI_API_KEY")

#Initialize LLM
llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)

C:\Users\mdfai\AppData\Local\Temp\ipykernel_20032\3725230459.py:6: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [3]:
#Create Vector DB (Retriever)
with open("sample.txt","r", encoding="utf-8")as f:
    text_data= f.read()

#Split the text into smaller chunks
splitter= CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts= splitter.split_text(text_data)
embedding= OpenAIEmbeddings()
vectorstore= FAISS.from_texts(texts, embedding)

C:\Users\mdfai\AppData\Local\Temp\ipykernel_20032\2455974220.py:8: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding= OpenAIEmbeddings()


In [4]:
#Get retrieves --> it does not generate

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Compressor using LLM
compressor= LLMChainExtractor.from_llm(llm)

# Create compression retriever
compression_retriever= ContextualCompressionRetriever(
    base_compressor= compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval
print("Contextual Compression Result:")
results= compression_retriever.get_relevant_documents("Who created LangChain?")
for doc in results:
    print("-", doc.page_content)

Contextual Compression Result:


C:\Users\mdfai\AppData\Local\Temp\ipykernel_20032\399285010.py:17: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results= compression_retriever.get_relevant_documents("Who created LangChain?")


- LangChain was created by Harrison Chase.


In [5]:
#Integration with ConversationalRetrievalChain- Without agent mode
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

memory= ConversationBufferMemory(memory_key="chat_history", return_messages=True)

#ConversationalRetrievalChain
rag_chain= ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever= compression_retriever,
    memory=memory
)

#Ask a few questions
print("ConversationalRetrievalChain:")
print(rag_chain.invoke({"question": "What is LangChain?"})["answer"])
print(rag_chain.invoke({"question": "Who created it?"})["answer"])

C:\Users\mdfai\AppData\Local\Temp\ipykernel_20032\1530232518.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory= ConversationBufferMemory(memory_key="chat_history", return_messages=True)


ConversationalRetrievalChain:
LangChain is a framework for building applications with large language models (LLMs). It was created by Harrison Chase and supports various features such as retrieval-augmented generation (RAG), agents, memory, and tools. LangChain is commonly used in applications like chatbots, document question and answer systems, and AI workflows.
LangChain was created by Harrison Chase.


In [6]:
#Integration into Agent (with Tool)

from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool

#Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": [memory.chat_memory.messages]
    })["answer"]

# Structured Tool
rag_tool= StructuredTool.from_function(
    name="Rag_Toool",
    description="Answer LangChain-related questions with context.",
    func=rag_tool_fn
)

# Agent with memory
agent= initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

#Ask via agent
print("\n🔹 Agent Conversation:")
print(agent.run("What is LangChain?"))
print(agent.run("Who created it?"))

C:\Users\mdfai\AppData\Local\Temp\ipykernel_20032\175404909.py:21: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent= initialize_agent(
C:\Users\mdfai\AppData\Local\Temp\ipykernel_20032\175404909.py:31: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print(agent.run("What is LangChain?"))



🔹 Agent Conversation:


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? No
AI: LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides various features such as retrieval-augmented generation (RAG), agents, memory, and tools, making it suitable for applications like chatbots, document question and answer systems, and AI workflows. It aims to simplify the development process and enhance the capabilities of LLMs in practical applications.
```

> Finished chain.
LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides various features such as retrieval-augmented generation (RAG), agents, memory, and tools, making it suitable for applications like chatbots, document question and answer systems, and AI workflows. It aims to simplify the development process and enhance the capabilities of LLMs in practical applications.
```


> Entering new AgentExe

In [8]:
#multiQuery
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.chains import RetrievalQA

# MultiQueryRetriever
multi_query_retriever= MultiQueryRetriever.from_llm(
    retriever= vectorstore.as_retriever(),
    llm=llm
)

# RetrievalQA Chain
rag_multi=RetrievalQA.from_chain_type(
    llm=llm,
    retriever=multi_query_retriever,
    return_source_documents=True
)

# 🧪 Ask question
print("\n🔹 RAG Pipeline (MultiQueryRetriever):")
res = rag_multi.invoke({"query": "Tell me about LangChain creator and features."})

print("Answer:", res["result"])
print("\nSources:")
for doc in res["source_documents"]:
    print("-", doc.page_content[:200])


🔹 RAG Pipeline (MultiQueryRetriever):
Answer: LangChain was created by Harrison Chase. It is a framework designed for building applications with large language models (LLMs). Some of its key features include support for retrieval-augmented generation (RAG), agents, memory, tools, and more. LangChain is commonly used in applications such as chatbots, document question and answer systems, and AI workflows.

Sources:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&


In [9]:
#Integration into Agent (with Tool)

#Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_multi.invoke({
        "question": question,
        "chat_history": [memory.chat_memory.messages]
    })["answer"]

# Structured Tool
rag_tool= StructuredTool.from_function(
    name="Rag_Toool",
    description="Answer LangChain-related questions with context.",
    func=rag_tool_fn
)

# Agent with memory
agent= initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

#Ask via agent
print("\n🔹 Agent Conversation:")
print(agent.run("What is LangChain?"))
print(agent.run("Who created it?"))


🔹 Agent Conversation:


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? No
AI: LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides various features such as retrieval-augmented generation (RAG), agents, memory, and tools, making it suitable for applications like chatbots, document question and answer systems, and AI workflows. It aims to simplify the development process and enhance the capabilities of LLMs in practical applications.
```

> Finished chain.
LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides various features such as retrieval-augmented generation (RAG), agents, memory, and tools, making it suitable for applications like chatbots, document question and answer systems, and AI workflows. It aims to simplify the development process and enhance the capabilities of LLMs in practical applications.
```


> Entering new AgentExe